# DockPulse — GPU Pipeline & Acceleration Benchmark

**Run this on Google Colab with a GPU.** `Runtime > Change runtime type > T4 GPU`.

This notebook is the *engine room*: it pulls London bike trips from **BigQuery**, runs the **same** `build_features()` function on **pandas (CPU)** and **cuDF (GPU)** to prove acceleration, scores station risk, and exports the two tiny artifacts the Streamlit app serves:
- `data/station_features.parquet`
- `data/benchmark.json`

**Four beats:** ① User + Data → ② Build Pipeline → ③ Add Acceleration → ④ Show Decision.

## 0. Setup — clone the repo & install cuDF
We reuse the repo's `src/features.py` so the CPU and GPU paths run *identical* code.

In [ ]:
# Clone your repo (edit if your GitHub username differs)
!git clone https://github.com/tanialapalelo/dockpulse.git 2>/dev/null || echo 'already cloned'
%cd dockpulse

# cuDF is preinstalled on Colab GPU runtimes. If not, uncomment:
# !pip install -q cudf-cu12 --extra-index-url=https://pypi.nvidia.com
import cudf, pandas as pd
print('cuDF', cudf.__version__, '| pandas', pd.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Beat ①②  —  Authenticate & pull trips from BigQuery
Set `PROJECT_ID` to your GCP project (the one with BigQuery enabled). The date window is bounded to stay in the free tier — **widen it for a bigger GPU win** if you have quota.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'YOUR_GCP_PROJECT_ID'   # <-- EDIT ME

from google.cloud import bigquery
from src.extract_bq import query_sql
from src import config

# Widen the window here for more rows (=> bigger speedup). Default ~3 months.
SQL = query_sql(config.BQ_START_DATE, config.BQ_END_DATE)
client = bigquery.Client(project=PROJECT_ID)

# Dry-run first so you see bytes billed before spending quota
dry = client.query(SQL, job_config=bigquery.QueryJobConfig(dry_run=True))
print(f'This query will scan ~{dry.total_bytes_processed/1e9:.2f} GB')

In [ ]:
pdf = client.query(SQL).result().to_dataframe(create_bqstorage_client=True)
# ensure datetime dtype for the .dt accessors used in build_features
pdf['start_date'] = pd.to_datetime(pdf['start_date'])
pdf['end_date'] = pd.to_datetime(pdf['end_date'])
print(f'Pulled {len(pdf):,} trips')
pdf.head(3)

## 2. Beat ③  —  The acceleration benchmark: pandas (CPU) vs cuDF (GPU)
Same `build_features()`. Only the dataframe library changes.

In [ ]:
import time
from src.features import build_features

# --- CPU path (pandas) ---
t0 = time.perf_counter()
feats_cpu = build_features(pdf)
cpu_seconds = time.perf_counter() - t0
print(f'CPU (pandas): {cpu_seconds:.2f}s  ->  {len(feats_cpu):,} feature rows')

In [ ]:
# --- GPU path (cuDF) ---
gdf = cudf.from_pandas(pdf)
cudf.get_default_stream().synchronize() if hasattr(cudf, 'get_default_stream') else None
t0 = time.perf_counter()
feats_gpu = build_features(gdf)
_ = feats_gpu.shape  # force materialisation
gpu_seconds = time.perf_counter() - t0
print(f'GPU (cuDF): {gpu_seconds:.2f}s  ->  {len(feats_gpu):,} feature rows')

In [ ]:
import json, datetime, subprocess

gpu_name = subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader').strip() or 'NVIDIA GPU'
n = len(pdf)
benchmark = {
    'n_trips': int(n),
    'cpu_seconds': round(cpu_seconds, 2),
    'gpu_seconds': round(gpu_seconds, 2),
    'speedup': round(cpu_seconds / max(gpu_seconds, 1e-6), 1),
    'rows_per_sec_cpu': int(n / cpu_seconds),
    'rows_per_sec_gpu': int(n / gpu_seconds),
    'gpu_name': gpu_name,
    'generated_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'is_synthetic': False,
}
with open('data/benchmark.json', 'w') as f:
    json.dump(benchmark, f, indent=2)
print(json.dumps(benchmark, indent=2))
print(f"\n>>> {benchmark['speedup']}x faster on {gpu_name} <<<")

## 3. Beat ④  —  Score risk & export the app artifact
Attach real dock capacity from `cycle_stations`, then save the compact feature table the app serves.

In [ ]:
# real dock capacity per station
docks = client.query(
    f"SELECT id AS station_id, docks_count FROM `{config.BQ_STATIONS_TABLE}`"
).result().to_dataframe()

features = feats_gpu.to_pandas() if hasattr(feats_gpu, 'to_pandas') else feats_gpu
features = features.merge(docks, on='station_id', how='left')
features['docks_count'] = features['docks_count'].fillna(20).astype(int)
features.to_parquet('data/station_features.parquet', index=False)
print(f'Saved {len(features):,} rows for {features.station_id.nunique()} stations')

# sanity: score the default demo hour
from src import score_risk
scored = score_risk.score(features, config.DEFAULT_HOUR_OF_WEEK)
print('At-risk this hour:', int((scored.status != 'OK').sum()))
score_risk.alerts(scored, 5)[['station_name','status','risk','bikes_to_move']]

## 4. Ship the artifacts back to the repo
Download the two files and commit them to `data/` (or push from Colab with a GitHub token). The Streamlit app auto-loads them.

In [ ]:
from google.colab import files
files.download('data/station_features.parquet')
files.download('data/benchmark.json')
print('Commit these into data/ then push. Streamlit Cloud will redeploy automatically.')